In [1]:
from google.colab import drive
import os
drive.mount('/content/drive')
path = '/content/drive/MyDrive/DM'
os.chdir(path)

Mounted at /content/drive


In [2]:
import pickle
import gzip
import logging
from collections import defaultdict, Counter
from typing import List, Tuple, Dict, Any, Union
from tqdm.auto import tqdm

# Cấu hình logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

class NgramLanguageModel:
    """
    Mô hình N-gram Language Model thống kê truyền thống.
    Hỗ trợ N-gram tổng quát (Bigram, Trigram...) và nhiều chiến lược Smoothing.
    """

    def __init__(self, n: int = 3, smoothing: str = 'laplace', k: float = 1.0, discount: float = 0.75):
        """
        :param n: Kích thước cửa sổ N-gram (2 cho Bigram, 3 cho Trigram).
        :param smoothing: Phương pháp làm mịn ('none', 'laplace', 'add-k', 'kneser-ney').
        :param k: Tham số k cho Add-k (Laplace mặc định k=1.0).
        :param discount: Tham số d (discount) cho Kneser-Ney (thường từ 0.5 đến 0.75).
        """
        assert n >= 2, "Model cần ít nhất n=2 (Bigram)"
        assert smoothing in ['none', 'laplace', 'add-k', 'kneser-ney'], "Smoothing không hợp lệ!"

        self.n = n
        self.smoothing = smoothing
        self.k = 1.0 if smoothing == 'laplace' else k
        self.discount = discount

        # Bảng tần suất
        self.vocab = set()
        self.vocab_size = 0

        # self.counts[context][target] = count
        self.counts = defaultdict(Counter)

        # self.context_totals[context] = tổng số lần context xuất hiện
        self.context_totals = Counter()

        # --- Dành riêng cho Kneser-Ney ---
        self.continuation_counts = Counter()
        self.total_ngram_types = 0

    # ==========================================
    # PHẦN 1 & 5: COUNT N-GRAM & OPTIMIZATION
    # ==========================================
    def fit(self, corpus: List[List[str]], vocab: set):
        """
        Huấn luyện mô hình bằng cách đếm tần suất N-gram từ corpus.
        """
        logger.info(f"Đang huấn luyện {self.n}-gram model với smoothing '{self.smoothing}'...")
        self.vocab = vocab
        self.vocab_size = len(self.vocab)

        for sentence in tqdm(corpus, desc="Counting N-grams"):
            if len(sentence) < self.n:
                continue

            # Trượt cửa sổ N-gram qua từng câu
            for i in range(len(sentence) - self.n + 1):
                window = sentence[i : i + self.n]
                context = tuple(window[:-1])
                target = window[-1]

                self.counts[context][target] += 1
                self.context_totals[context] += 1

                if self.smoothing == 'kneser-ney':
                    # Đếm số lượng context khác nhau đứng trước target w
                    # Cần thiết để tính P_continuation
                    self.continuation_counts[target] += 1
                    self.total_ngram_types += 1

        # Memory Optimization: Convert defaultdict về dict chuẩn để giảm RAM
        logger.info("Đang tối ưu hóa bộ nhớ (Freezing dicts)...")
        self.counts = {ctx: dict(target_counts) for ctx, target_counts in self.counts.items()}
        self.context_totals = dict(self.context_totals)

        if self.smoothing == 'kneser-ney':
            self.continuation_counts = dict(self.continuation_counts)

        logger.info(f"Huấn luyện xong! Ghi nhận {len(self.counts):,} unique contexts.")

    # ==========================================
    # PHẦN 2 & 3: PROBABILITY & SMOOTHING
    # ==========================================
    def predict_proba(self, context: Tuple[str, ...], target: str) -> float:
        """
        Tính xác suất P(target | context).
        """
        # Fallback an toàn nếu context hoặc target chứa từ không có trong vocab
        safe_target = target if target in self.vocab else '<UNK>'
        safe_context = tuple([w if w in self.vocab else '<UNK>' for w in context])

        # Nếu context nhập vào dài hơn thiết lập của mô hình, cắt bớt lấy phần đuôi
        if len(safe_context) > self.n - 1:
            safe_context = safe_context[-(self.n - 1):]
        # Nếu context nhập vào ngắn hơn, đệm thêm <START>
        elif len(safe_context) < self.n - 1:
            pad_len = (self.n - 1) - len(safe_context)
            safe_context = tuple(['<START>'] * pad_len) + safe_context

        count_c_w = self.counts.get(safe_context, {}).get(safe_target, 0)
        count_c = self.context_totals.get(safe_context, 0)

        # 1. No Smoothing (Maximum Likelihood)
        if self.smoothing == 'none':
            if count_c == 0: return 0.0
            return count_c_w / count_c

        # 2 & 3. Laplace / Add-k Smoothing
        elif self.smoothing in ['laplace', 'add-k']:
            return (count_c_w + self.k) / (count_c + self.k * self.vocab_size)

        # 4. Kneser-Ney Smoothing (Interpolated)
        elif self.smoothing == 'kneser-ney':
            # Xác suất Continuation
            p_cont = self.continuation_counts.get(safe_target, 0) / max(1, self.total_ngram_types)

            if count_c == 0:
                # Nếu context chưa từng xuất hiện, trả về hoàn toàn xác suất continuation
                return p_cont

            # Tính Discounted Probability
            discounted_prob = max(count_c_w - self.discount, 0) / count_c

            # Tính Lambda (Trọng số nội suy)
            unique_continuations = len(self.counts.get(safe_context, {}))
            lambda_weight = (self.discount / count_c) * unique_continuations

            return discounted_prob + lambda_weight * p_cont

    # ==========================================
    # PREDICT NEXT (Ứng dụng)
    # ==========================================
    def predict_next(self, context: Tuple[str, ...], top_k: int = 5) -> List[Tuple[str, float]]:
        """
        Dự đoán K từ tiếp theo có xác suất cao nhất.
        Thực hiện brute-force tính xác suất trên toàn bộ Vocabulary.
        """
        probabilities = []
        for word in self.vocab:
            # Bỏ qua token không mang ý nghĩa sinh văn bản
            if word in ['<START>', '<UNK>']:
                continue

            prob = self.predict_proba(context, word)
            probabilities.append((word, prob))

        # Sort giảm dần theo xác suất
        probabilities.sort(key=lambda x: x[1], reverse=True)
        return probabilities[:top_k]

    # ==========================================
    # PHẦN 6: SERIALIZATION (LƯU TRỮ)
    # ==========================================
    def save(self, filepath: str, compressed: bool = True):
        """Lưu model. Dùng gzip để nén vì count dictionary rất tốn dung lượng."""
        logger.info(f"Đang lưu model tại {filepath} (Compressed: {compressed})...")
        open_func = gzip.open if compressed else open

        with open_func(filepath, 'wb') as f:
            pickle.dump(self.__dict__, f, protocol=pickle.HIGHEST_PROTOCOL)
        logger.info("Đã lưu xong!")

    @classmethod
    def load(cls, filepath: str, compressed: bool = True) -> 'NgramLanguageModel':
        """Load model từ file."""
        logger.info(f"Đang load model từ {filepath}...")
        open_func = gzip.open if compressed else open

        with open_func(filepath, 'rb') as f:
            data = pickle.load(f)

        # Khôi phục instance
        model = cls(n=data['n'], smoothing=data['smoothing'], k=data['k'], discount=data['discount'])
        model.__dict__.update(data)

        logger.info("Load model thành công!")
        return model

In [ ]:
# 1. Load Data đã preprocessed
with open('/content/drive/MyDrive/DM/data/train/train.pkl', 'rb') as f:
    train_corpus = pickle.load(f)
with open('/content/drive/MyDrive/DM/data/train/vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)

# 2. Khởi tạo và Train (Ví dụ: Trigram với Laplace Smoothing)
model = NgramLanguageModel(n=3, smoothing='laplace')
model.fit(train_corpus, vocab)

# 3. Dự đoán (Context là 2 từ trước đó do n=3)
context = ('tôi', 'đang')
top_words = model.predict_next(context, top_k=5)

print(f"Ngữ cảnh: {context}")
for word, prob in top_words:
    print(f" -> {word}: {prob:.6f}")

# 4. Lưu lại
model.save('/content/drive/MyDrive/DM/data/train/trigram_laplace.pkl.gz')

Counting N-grams:   0%|          | 0/893309 [00:00<?, ?it/s]

Ngữ cảnh: ('tôi', 'đang')
 -> cố_gắng: 0.000078
 -> ở: 0.000067
 -> làm: 0.000039
 -> làm_việc: 0.000034
 -> rất: 0.000028


In [ ]:
# Kneser-Ney với hệ số discount mặc định (0.75)
model_kn = NgramLanguageModel(n=3, smoothing='kneser-ney')
model_kn.fit(train_corpus, vocab)

# Dự đoán
context = ('tôi', 'đang')
top_words = model_kn.predict_next(context, top_k=5)
print(f"Ngữ cảnh: {context}")
for word, prob in top_words:
    print(f" -> {word}: {prob:.6f}")

model_kn.save('/content/drive/MyDrive/DM/data/train/trigram_kn.pkl.gz')

Counting N-grams:   0%|          | 0/893309 [00:00<?, ?it/s]

Ngữ cảnh: ('tôi', 'đang')
 -> cố_gắng: 0.081706
 -> ở: 0.071172
 -> làm: 0.035947
 -> làm_việc: 0.028508
 -> có: 0.024188


In [ ]:
# Add-k Smoothing với hệ số k tùy chỉnh (ví dụ: 0.01 để không làm xác suất bị giảm quá mạnh như Laplace)
model_add_k = NgramLanguageModel(n=3, smoothing='add-k', k=0.01)
model_add_k.fit(train_corpus, vocab)

# Dự đoán
context = ('tôi', 'đang')
top_words = model_add_k.predict_next(context, top_k=5)
print(f"Ngữ cảnh: {context}")
for word, prob in top_words:
    print(f" -> {word}: {prob:.6f}")

# Lưu mô hình
model_add_k.save('/content/drive/MyDrive/DM/data/train/trigram_add_k.pkl.gz')

Counting N-grams:   0%|          | 0/893309 [00:00<?, ?it/s]

Ngữ cảnh: ('tôi', 'đang')
 -> cố_gắng: 0.006733
 -> ở: 0.005698
 -> làm: 0.003110
 -> làm_việc: 0.002593
 -> rất: 0.002075


# Cải tiến mô hình với Modified Kneser-Ney


In [3]:
class AdvancedNgramLanguageModel(NgramLanguageModel):
    """
    Kế thừa từ mô hình N-gram cũ, bổ sung thêm Modified Kneser-Ney (MKN)
    và chuẩn bị cơ sở cho Hybrid Model.
    """
    def __init__(self, n: int = 3, smoothing: str = 'modified-kneser-ney', k: float = 1.0, discount: float = 0.75):
        super().__init__(n=n, smoothing='kneser-ney', k=k, discount=discount)

        # Ghi đè lại tên smoothing thực sự của class con
        self.smoothing = smoothing

        # Các tham số Adaptive Discounting cho MKN
        self.D1 = 0.0
        self.D2 = 0.0
        self.D3plus = 0.0

    def fit(self, corpus: List[List[str]], vocab: set):
        logger.info(f"Đang huấn luyện {self.n}-gram model với smoothing '{self.smoothing}' (Advanced Class)...")
        self.vocab = vocab
        self.vocab_size = len(self.vocab)

        # 1. Quét corpus để đếm N-gram như bình thường
        for sentence in tqdm(corpus, desc="Counting N-grams (Pass 1)"):
            if len(sentence) < self.n: continue
            for i in range(len(sentence) - self.n + 1):
                window = sentence[i : i + self.n]
                context = tuple(window[:-1])
                target = window[-1]

                self.counts[context][target] += 1
                self.context_totals[context] += 1

                if self.smoothing in ['kneser-ney', 'modified-kneser-ney']:
                    self.continuation_counts[target] += 1
                    self.total_ngram_types += 1

        # 2. Xử lý toán học riêng cho Modified Kneser-Ney
        if self.smoothing == 'modified-kneser-ney':
            n1 = n2 = n3 = n4 = 0

            logger.info("Đang tính toán Adaptive Discounting (D1, D2, D3+)...")
            for ctx, targets in tqdm(self.counts.items(), desc="Calculating MKN Parameters"):
                for w, c in targets.items():
                    if c == 1: n1 += 1
                    elif c == 2: n2 += 1
                    elif c == 3: n3 += 1
                    elif c == 4: n4 += 1

            # Tránh lỗi chia 0
            n1 = max(n1, 1); n2 = max(n2, 1); n3 = max(n3, 1)

            # Công thức Chen & Goodman (1998)
            Y = n1 / (n1 + 2 * n2)
            self.D1 = 1 - 2 * Y * (n2 / n1)
            self.D2 = 2 - 3 * Y * (n3 / n2)
            self.D3plus = 3 - 4 * Y * (n4 / n3)

            logger.info(f"Adaptive Parameters: D1={self.D1:.4f}, D2={self.D2:.4f}, D3+={self.D3plus:.4f}")

        # 3. Tối ưu bộ nhớ (Giống code cũ)
        logger.info("Đang tối ưu hóa RAM (Freezing dicts)...")
        self.counts = {ctx: dict(target_counts) for ctx, target_counts in self.counts.items()}
        self.context_totals = dict(self.context_totals)
        if self.smoothing in ['kneser-ney', 'modified-kneser-ney']:
            self.continuation_counts = dict(self.continuation_counts)
        logger.info("Huấn luyện hoàn tất!")

    def predict_proba(self, context: Tuple[str, ...], target: str) -> float:
        # Xử lý context và target an toàn (giống code cũ)
        safe_target = target if target in self.vocab else '<UNK>'
        safe_context = tuple([w if w in self.vocab else '<UNK>' for w in context])

        if len(safe_context) > self.n - 1:
            safe_context = safe_context[-(self.n - 1):]
        elif len(safe_context) < self.n - 1:
            pad_len = (self.n - 1) - len(safe_context)
            safe_context = tuple(['<START>'] * pad_len) + safe_context

        count_c_w = self.counts.get(safe_context, {}).get(safe_target, 0)
        count_c = self.context_totals.get(safe_context, 0)

        # Trả về nhánh code cũ nếu không phải MKN
        if self.smoothing != 'modified-kneser-ney':
            return super().predict_proba(context, target)

        # --- Nhánh xử lý riêng cho Modified Kneser-Ney ---
        p_cont = self.continuation_counts.get(safe_target, 0) / max(1, self.total_ngram_types)

        if count_c == 0:
            return p_cont

        # Lựa chọn mức Discount thích ứng
        if count_c_w == 0:
            discount = 0.0
        elif count_c_w == 1:
            discount = self.D1
        elif count_c_w == 2:
            discount = self.D2
        else:
            discount = self.D3plus

        discounted_prob = max(count_c_w - discount, 0) / count_c

        # Tính toán nội suy Lambda chính xác hơn
        n1_c = n2_c = n3plus_c = 0
        for w, c in self.counts.get(safe_context, {}).items():
            if c == 1: n1_c += 1
            elif c == 2: n2_c += 1
            elif c >= 3: n3plus_c += 1

        lambda_weight = (self.D1 * n1_c + self.D2 * n2_c + self.D3plus * n3plus_c) / count_c

        return discounted_prob + lambda_weight * p_cont

In [5]:
# Khởi tạo mô hình Modified Kneser-Ney (MKN)
model_mkn = AdvancedNgramLanguageModel(n=3, smoothing='modified-kneser-ney')

# Train mô hình (Sử dụng lại train_corpus và vocab đã load ở các cell trên)
model_mkn.fit(train_corpus, vocab)

# Test dự đoán thử
context = ('tôi', 'đang')
top_words = model_mkn.predict_next(context, top_k=5)

print(f"\nNgữ cảnh: {context} (Modified Kneser-Ney)")
for word, prob in top_words:
    print(f" -> {word}: {prob:.6f}")

# Lưu mô hình để chuẩn bị đưa sang evaluation.ipynb so sánh
save_path = '/content/drive/MyDrive/DM/data/train/trigram_mkn.pkl.gz'
model_mkn.save(save_path)

Counting N-grams (Pass 1):   0%|          | 0/893309 [00:00<?, ?it/s]

Calculating MKN Parameters:   0%|          | 0/3998353 [00:00<?, ?it/s]


Ngữ cảnh: ('tôi', 'đang') (Modified Kneser-Ney)
 -> cố_gắng: 0.077393
 -> ở: 0.067588
 -> làm: 0.031871
 -> làm_việc: 0.024231
 -> <END>: 0.021996


In [6]:
import time

# 1. Khởi tạo và Train mô hình Bigram (n=2)
print("\nBẮT ĐẦU TRAIN BIGRAM MODIFIED KNESER-NEY")
start_time = time.time()
model_bigram_mkn = AdvancedNgramLanguageModel(n=2, smoothing='modified-kneser-ney')
model_bigram_mkn.fit(train_corpus, vocab)
print(f"Thời gian train: {time.time() - start_time:.2f}s")

# 2. Lưu mô hình Bigram
save_path_bigram = '/content/drive/MyDrive/DM/data/train/bigram_mkn.pkl.gz'
model_bigram_mkn.save(save_path_bigram)
print(f"Đã lưu mô hình Bigram tại: {save_path_bigram}")


BẮT ĐẦU TRAIN BIGRAM MODIFIED KNESER-NEY


Counting N-grams (Pass 1):   0%|          | 0/893309 [00:00<?, ?it/s]

Calculating MKN Parameters:   0%|          | 0/174481 [00:00<?, ?it/s]

Thời gian train: 63.99s
Đã lưu mô hình Bigram tại: /content/drive/MyDrive/DM/data/train/bigram_mkn.pkl.gz
